In [1]:
# %% Cell 1 — Load libraries
import sqlite3
import pandas as pd

df = pd.read_parquet('../data/processed/cleaned.parquet')
df['date'] = df['date'].astype(str)  # SQLite doesn't support datetime natively

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

Loaded 5477 rows
Columns: ['date', 'rides', 'temp_mean', 'temp_max', 'temp_min', 'precipitation', 'windspeed', 'weather_code', 'weather_desc', 'is_holiday', 'year', 'month', 'day_of_week', 'is_weekend', 'season', 'season_name', 'is_rainy', 'day_type', 'rides_vs_yearly_avg']


In [2]:
# %% Cell 2 — Create SQLite database and store the data
# SQLite is a lightweight file-based SQL database
# No server needed — the database is just a single file
# This demonstrates the Storage lecture content (Week 3)

conn = sqlite3.connect('../data/processed/london_bikes.db')

# Write the entire cleaned dataframe to a SQL table called 'rides'
df.to_sql('rides', conn, if_exists='replace', index=False)

print("Database created: data/processed/london_bikes.db")
print("Table 'rides' created with all cleaned data")

# Verify it worked
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM rides")
print(f"Rows in database: {cursor.fetchone()[0]}")

Database created: data/processed/london_bikes.db
Table 'rides' created with all cleaned data
Rows in database: 5477


In [3]:
# %% Cell 3 — SQL Query 1: Average rides by day type
# This is the core finding of our research question
# Same result as notebook 05 but done in SQL

query1 = """
SELECT 
    day_type,
    COUNT(*) as num_days,
    ROUND(AVG(rides), 0) as avg_rides,
    ROUND(MIN(rides), 0) as min_rides,
    ROUND(MAX(rides), 0) as max_rides
FROM rides
GROUP BY day_type
ORDER BY avg_rides DESC
"""

result1 = pd.read_sql_query(query1, conn)
print("=== AVERAGE RIDES BY DAY TYPE ===")
print(result1.to_string(index=False))

=== AVERAGE RIDES BY DAY TYPE ===
day_type  num_days  avg_rides  min_rides  max_rides
 Weekday      3788    28077.0      188.0    73094.0
 Weekend      1564    23445.0     3531.0    70170.0
 Holiday       125    20545.0     4327.0    67034.0


In [4]:
# %% Cell 4 — SQL Query 2: Average rides and temperature by season
query2 = """
SELECT
    season_name,
    COUNT(*) as num_days,
    ROUND(AVG(rides), 0) as avg_rides,
    ROUND(AVG(temp_mean), 1) as avg_temp_celsius
FROM rides
GROUP BY season_name
ORDER BY avg_rides DESC
"""

result2 = pd.read_sql_query(query2, conn)
print("=== AVERAGE RIDES AND TEMPERATURE BY SEASON ===")
print(result2.to_string(index=False))

=== AVERAGE RIDES AND TEMPERATURE BY SEASON ===
season_name  num_days  avg_rides  avg_temp_celsius
     Summer      1380    33296.0              17.4
     Autumn      1363    27715.0              11.9
     Spring      1380    26359.0               9.9
     Winter      1354    18828.0               5.6


In [5]:
# %% Cell 5 — SQL Query 3: Top 10 busiest days ever
query3 = """
SELECT 
    date,
    rides,
    ROUND(temp_mean, 1) as temp_celsius,
    weather_desc,
    day_type,
    season_name
FROM rides
ORDER BY rides DESC
LIMIT 10
"""

result3 = pd.read_sql_query(query3, conn)
print("=== TOP 10 BUSIEST DAYS EVER ===")
print(result3.to_string(index=False))

=== TOP 10 BUSIEST DAYS EVER ===
      date  rides  temp_celsius  weather_desc day_type season_name
2015-07-09  73094          15.9 Partly cloudy  Weekday      Summer
2020-05-30  70170          16.4 Partly cloudy  Weekend      Spring
2020-05-25  67034          17.3 Partly cloudy  Holiday      Spring
2022-08-19  66972          20.1       Drizzle  Weekday      Summer
2022-06-21  65326          16.5 Partly cloudy  Weekday      Summer
2020-06-13  65045          17.1          Rain  Weekend      Summer
2020-06-20  64041          16.3       Drizzle  Weekend      Summer
2015-08-06  63963          17.7       Drizzle  Weekday      Summer
2020-05-31  63116          17.4         Clear  Weekend      Spring
2020-06-14  57516          16.7       Drizzle  Weekend      Summer


In [6]:
# %% Cell 6 — SQL Query 4: Top 10 quietest days (excluding anomalies)
query4 = """
SELECT
    date,
    rides,
    ROUND(temp_mean, 1) as temp_celsius,
    weather_desc,
    day_type,
    season_name
FROM rides
WHERE rides > 0
ORDER BY rides ASC
LIMIT 10
"""

result4 = pd.read_sql_query(query4, conn)
print("=== TOP 10 QUIETEST DAYS ===")
print(result4.to_string(index=False))

=== TOP 10 QUIETEST DAYS ===
      date  rides  temp_celsius  weather_desc day_type season_name
2025-08-05    188          17.1 Partly cloudy  Weekday      Summer
2012-02-05   3531           0.4          Snow  Weekend      Winter
2013-01-20   3728          -1.3          Snow  Weekend      Winter
2014-01-01   4327           8.0          Rain  Holiday      Winter
2011-01-01   4555           4.5       Drizzle  Weekend      Winter
2013-09-15   4826          11.3       Drizzle  Weekend      Autumn
2020-04-28   4872           8.9          Rain  Weekday      Spring
2011-12-24   4890           4.9       Drizzle  Weekend      Winter
2016-01-03   4894           7.7          Rain  Weekend      Winter
2013-02-10   4901           2.6          Snow  Weekend      Winter


In [7]:
# %% Cell 7 — SQL Query 5: Combined effect of holidays and rain
# Does being both a holiday AND rainy make things even worse?
query5 = """
SELECT
    CASE WHEN is_holiday = 1 THEN 'Holiday' ELSE 'Non-holiday' END as holiday_status,
    CASE WHEN is_rainy = 1 THEN 'Rainy' ELSE 'Dry' END as rain_status,
    COUNT(*) as num_days,
    ROUND(AVG(rides), 0) as avg_rides
FROM rides
GROUP BY is_holiday, is_rainy
ORDER BY avg_rides DESC
"""

result5 = pd.read_sql_query(query5, conn)
print("=== COMBINED EFFECT OF HOLIDAYS AND RAIN ===")
print(result5.to_string(index=False))

=== COMBINED EFFECT OF HOLIDAYS AND RAIN ===
holiday_status rain_status  num_days  avg_rides
   Non-holiday         Dry      3496    28685.0
       Holiday         Dry        85    23442.0
   Non-holiday       Rainy      1856    23028.0
       Holiday       Rainy        40    14388.0


In [8]:
# %% Cell 8 — SQL Query 6: Holiday effect broken down by season
# Are holidays more damaging to ridership in summer or winter?
query6 = """
SELECT
    season_name,
    day_type,
    COUNT(*) as num_days,
    ROUND(AVG(rides), 0) as avg_rides
FROM rides
GROUP BY season_name, day_type
ORDER BY season_name, avg_rides DESC
"""

result6 = pd.read_sql_query(query6, conn)
print("=== HOLIDAY EFFECT BY SEASON ===")
print(result6.to_string(index=False))

=== HOLIDAY EFFECT BY SEASON ===
season_name day_type  num_days  avg_rides
     Autumn  Weekday       974    29334.0
     Autumn  Weekend       388    23662.0
     Autumn  Holiday         1    23197.0
     Spring  Weekday       926    27341.0
     Spring  Weekend       394    24445.0
     Spring  Holiday        60    23783.0
     Summer  Weekday       966    34285.0
     Summer  Weekend       395    31178.0
     Summer  Holiday        19    27048.0
     Winter  Weekday       922    20985.0
     Winter  Weekend       387    14318.0
     Winter  Holiday        45    13423.0


In [9]:
# %% Cell 9 — SQL Query 7: Yearly summary
query7 = """
SELECT
    year,
    COUNT(*) as num_days,
    ROUND(AVG(rides), 0) as avg_rides,
    MAX(rides) as max_rides,
    MIN(rides) as min_rides,
    ROUND(AVG(temp_mean), 1) as avg_temp
FROM rides
GROUP BY year
ORDER BY year
"""

result7 = pd.read_sql_query(query7, conn)
print("=== YEARLY SUMMARY ===")
print(result7.to_string(index=False))

conn.close()
print("\nSQL notebook complete! Database saved to data/processed/london_bikes.db")

=== YEARLY SUMMARY ===
 year  num_days  avg_rides  max_rides  min_rides  avg_temp
 2011       365    19568.0      29417       4555      11.3
 2012       366    26009.0      47102       3531      10.1
 2013       365    22042.0      35580       3728       9.9
 2014       365    27463.0      49025       4327      11.5
 2015       365    27046.0      73094       5779      11.0
 2016       366    28152.0      46625       4894      10.9
 2017       365    28619.0      46035       5143      11.2
 2018       365    28952.0      46084       5859      11.5
 2019       365    28562.0      44668       5649      11.1
 2020       366    28509.0      70170       4872      11.6
 2021       365    29976.0      56922       6253      10.8
 2022       363    31697.0      66972       5853      11.9
 2023       365    23373.0      35319       6721      11.6
 2024       366    23957.0      35190       7479      11.7
 2025       365    24840.0      45823        188      12.4

SQL notebook complete! Database 